# Computational Derivations and Spectral Invariants of Girard Torsion, Jordan Algebra H3(O), and 3-Torus Eigenmodes Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.k1n2-dzgt/fair2.json
```

The Croissant schema defines entities (record sets, fields, columns, etc.) uniquely using their `@id` fields. Throughout this notebook, we reference all schema elements by their `@id`.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and overview records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.k1n2-dzgt/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript or iterate)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Identifier: {meta.identifier}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Each record set, field, and column in the dataset is uniquely defined by its `@id`. Below, we enumerate all available record sets and their fields. This helps map out the data structure for later steps.

In [ ]:
# List record sets defined in the dataset
record_sets = [rs['@id'] for rs in meta.recordSet] if hasattr(meta, 'recordSet') and meta.recordSet else []

if len(record_sets) == 0:
    print("No record sets found in the metadata.")
else:
    print("Record Sets (by @id):")
    for rs_id in record_sets:
        print(f"- {rs_id}")
    # Inspect fields within each record set
    for rs in meta.recordSet:
        print(f"\nFields in RecordSet {rs['@id']}:")
        for field in rs['field']:
            print(f"  - {field['@id']} (type: {field.get('dataType', 'unknown')})")
else:
    # For demonstration, try loading records directly by an example record set id
    example_record_set_id = None  # will be set later

## 3. Data Extraction
Load tabular data from one or more record sets into DataFrames for analysis.

If the schema contains record sets, we extract all records using their record set `@id`. All data element references use `@id` in variables and function calls.

In [ ]:
# Collect record set ids for extraction
if len(record_sets) == 0:
    print("No record sets to extract.")
    # Try example id if known; in practice, you would inspect the Croissant schema
    # e.g., example_record_set_id = 'cr:RecordSet/derivationRecords'
else:
    dataframes = {}
    for rs_id in record_sets:
        print(f"Loading records from RecordSet @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Columns for {rs_id}: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for {rs_id}.")

# For demonstration, if no record sets are available (empty dict returned), try using a placeholder
# NOTE: You should update <record_set_id> and <field_id> with actual @id values from step 2

## 4. Exploratory Data Analysis (EDA)
Perform common data processing steps: filtering, normalizing, grouping, and removing outliers.

We use `@id` fields to reference numeric and grouping fields. Adjust these values based on the actual schema structure you found in step 2.

In [ ]:
# Example EDA workflow using @id references
# Please replace these ids with actual field ids identified in step 2

# Try extracting a RecordSet to demonstrate
if len(dataframes) > 0:
    # Select the first available RecordSet
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Identify a numeric field by its @id
    potential_numeric = [col for col in df.columns if df[col].dtype != 'O']
    if not potential_numeric:
        print("No numeric fields found.")
        numeric_field_id = None
    else:
        numeric_field_id = potential_numeric[0]
        print(f"Using numeric field @id: {numeric_field_id}")

    # Filtering based on numeric_field
    threshold = df[numeric_field_id].quantile(0.75) if numeric_field_id else None
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field (choose a non-numeric one)
        potential_groups = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field_id]
        group_field_id = potential_groups[0] if potential_groups else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No extracted dataframes to perform EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. Adjust the field `@id` variables to use ids from your specific schema.

In [ ]:
# Example visualization for extracted data
# Adjust numeric_field_id and group_field_id with your schema's ids
if len(dataframes) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No available numeric fields for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load and review metadata from a Croissant schema using `mlcroissant`.
- Enumerate available record sets and fields using their unique `@id`s.
- Extract tabular data and reference fields dynamically by `@id`.
- Perform basic EDA: filtering, normalization, and grouping based on key attributes.
- Visualize numeric distributions and relationships using `matplotlib` and `seaborn`.

For further analysis:
- Consult the Croissant schema for more specific field and record set definitions.
- Use `@id` references in all data access and transformation steps to ensure reproducibility and precise provenance mapping.